In [1]:
import os
import numpy as np
import re
import pandas as pd

In [32]:
def list_rv_filenames(toi_id, dir_prefix='toi', super_path='.'):
    '''
    List all RV filenames for a given TOI ID.

    Parameters:
    toi_id (str): The TOI ID to search for.
    dir_prefix (str): The prefix for the directory names. This includes anything that precedes the TOI ID and assumes that nothing follows.
    super_path (str): The base path where the TOI directories are located. Defaults to the current directory.

    Returns:
    list: A list of RV filenames associated with the TOI ID.
    '''
    dirname = os.path.join(super_path, dir_prefix + toi_id)

    if not os.path.exists(dirname):
        return []
    
    rv_filenames = [f for f in os.listdir(dirname) if f.endswith('.rv')]
    return rv_filenames

def find_num_rvs(rv_filename, toi_id, dir_prefix='toi', super_path='.'):
    '''
    Find the number of RVs in a given RV file.

    Parameters:
    rv_filename (str): The RV filename to read.
    toi_id (str): The TOI ID associated with the RV filename.
    dir_prefix (str): The prefix for the directory names. This includes anything that precedes the TOI ID and assumes that nothing follows.
    super_path (str): The base path where the TOI directories are located. Defaults to the current directory.

    Returns:
    int: The number of RVs in the file.
    '''
    # the following is assuming fit directories are named as e.g., 'toi7574' for TOI-7574
    path = os.path.join(super_path, dir_prefix + toi_id, rv_filename)

    if not os.path.exists(path):
        return 0
    
    with open(path, 'r') as file:
        lines = [line for line in file if not line.startswith('#')]
    
    # Assuming each line corresponds to an RV measurement
    num_rvs = len(lines)
    return num_rvs

def find_num_rvs_from_list(toi_id_list, dir_prefix='toi', super_path='.'):
    '''
    Find the number of RVs for a list of TOI IDs, separated by instrument.

    Parameters:
    toi_id_list (list): A list of TOI IDs to search for.
    dir_prefix (str): The prefix for the directory names. This includes anything that precedes the TOI ID and assumes that nothing follows.
    super_path (str): The base path where the TOI directories are located. Defaults to the current directory.

    Returns:
    dict: A dictionary with TOI IDs as keys and the number of RVs as values.
    '''
    num_rvs_dict = {}

    for toi_id in toi_id_list:
        rv_filenames = list_rv_filenames(toi_id, dir_prefix=dir_prefix, super_path=super_path)
        rv_instruments = [re.search(r'\d+\.(\w+)\.rv', filename).group(1) for filename in rv_filenames]

        num_rvs_dict[toi_id] = {instrument: find_num_rvs(filename, toi_id, dir_prefix=dir_prefix, super_path=super_path) for instrument, filename in zip(rv_instruments, rv_filenames)}


    return num_rvs_dict

def generate_rv_table(toi_id_list, dir_prefix = 'toi', super_path = '.'):
    '''
    Generate a LaTeX table including the number of RVs, median RV uncertainty, and first and last observation dates for each instrument and TOI.

    Parameters:
    toi_id_list (list): A list of TOI IDs to include in the table.
    dir_prefix (str): The prefix for the directory names. This includes anything that precedes the TOI ID and assumes that nothing follows.
    super_path (str): The base path where the TOI directories are located. Defaults to the current directory.

    Returns:
    str: A string containing the LaTeX table.
    '''

    # Create a DataFrame to hold the data
    data = []

    for toi_id in toi_id_list:
        rv_filenames = list_rv_filenames(toi_id, dir_prefix=dir_prefix, super_path=super_path)
        for filename in rv_filenames:
            instrument = re.search(r'\d+\.(\w+)\.rv', filename).group(1)
            num_rvs = find_num_rvs(filename, toi_id, dir_prefix=dir_prefix, super_path=super_path)

            # Read the RV file to get uncertainties and dates
            path = os.path.join(super_path, dir_prefix + toi_id, filename)
            df = pd.read_csv(path, comment='#', sep=r'\s+', header=None, names=['BJD', 'rv', 'sigma_rv'])
            
            median_uncertainty = df['sigma_rv'].median()
            first_obs_date = df['BJD'].min()
            last_obs_date = df['BJD'].max()

            # convert BJD to UTC calendar date
            first_obs_date_utc = pd.to_datetime(first_obs_date, unit='D', origin='julian').strftime('%d %b %Y')
            last_obs_date_utc = pd.to_datetime(last_obs_date, unit='D', origin='julian').strftime('%d %b %Y')

            data.append([toi_id, instrument, num_rvs, median_uncertainty, first_obs_date_utc, last_obs_date_utc])

    # Create a DataFrame from the collected data
    df_table = pd.DataFrame(data, columns=['TOI', 'Instrument', r'N$_{\text{RV}}$', r'Median $\sigma_{\text{RV}}$ (m/s)', 
                                           'First Observation \nDate (UT)', 'Last Observation \nDate (UT)'])
    
    # Make repeats of each TOI ''
    for toi_id in df_table['TOI'].unique():
        mask = df_table['TOI'] == toi_id
        df_table.loc[mask, 'TOI'] = [toi_id] + [''] * (mask.sum() - 1)

    # Generate LaTeX table
    latex_table = df_table.to_latex(index=False, float_format='%.1f', caption='Summary of RV Observations', label='tab:rv_summary')
    # change table env for multicolumn
    latex_table = latex_table.replace(r'\begin{table}', r'\begin{table*}').replace(r'\end{table}', r'\end{table*}') 
    # add a note below the table
    latex_table = latex_table.replace(r'\end{table*}', r'''\vspace{2mm}

\begin{minipage}{0.95\linewidth} 
\footnotesize
\textbf{Note:} The full table of RVs for each system is available in machine-readable form in the online journal.
\end{minipage} 
\end{table*}''')

    with open('rv_summary_table.tex', 'w') as f:
        f.write(latex_table)

In [33]:
toi_ids = ['3041', '3365', '3601', '3788', '3972', '3988', '3998', '4009', '4079', '4088', '4140', '4144', '5236', '5432', '5479', '5925',
            '6148', '6166', '6171', '6191', '6184', '6208', '6334', '6417', '6443', '7219', '7266', '7404', '7425', '7574']
dict = find_num_rvs_from_list(toi_ids)

In [34]:
tot_tres_spectra = sum(dict[toi_id].get('TRES', 0) for toi_id in toi_ids)

In [35]:
tot_tres_spectra

552

In [36]:
generate_rv_table(toi_ids)